## 图结构对象创建与使用
### 图结构概念
LangGraph的宗旨是创建一个图结构，该图结构包含大模型、外部工具等，通过点线间的连接构成灵活的处理链路。基于该宗旨，LangGraph定义了一套由点、边、状态组成的有向有环的结构图语法。
### 手动创建图流程

In [1]:
from langgraph.constants import START, END
from langgraph.graph import StateGraph
builder = StateGraph(dict)

def addition(state):
    """
    执行加法运算的节点函数

    参数:
        state (dict): 包含输入数据的状态字典，必须包含键"x"

    返回:
        dict: 返回更新后的状态字典，其中"x"的值增加1
    """
    print(f'加法节点收到的初始值:{state}')
    return {"x": state["x"] + 1}

def subtraction(state):
    """
    执行减法运算的节点函数

    参数:
        state (dict): 包含输入数据的状态字典，必须包含键"x"

    返回:
        dict: 返回更新后的状态字典，其中"x"的值减少2
    """
    print(f'减法节点收到的初始值:{state}')
    return {"x": state["x"] - 2}

# 向图构建器中添加节点
# 添加加法运算节点和减法运算节点到构建器中
builder.add_node("addition", addition)
builder.add_node("subtraction", subtraction)

# 定义节点之间的执行顺序 edges
# 设置节点间的依赖关系，形成执行流程图
builder.add_edge(START, "addition")
builder.add_edge("addition", "subtraction")
builder.add_edge("subtraction", END)
# 编译图构建器生成计算图
graph = builder.compile()

# 打印图的边和节点信息
print(builder.edges)
print(builder.nodes)
# 打印图的可视化结构
print(graph.get_graph().print_ascii())

{('__start__', 'addition'), ('subtraction', '__end__'), ('addition', 'subtraction')}
{'addition': StateNodeSpec(runnable=addition(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None, input_schema=<class 'dict'>, retry_policy=None, cache_policy=None, is_error_handler=False, error_handler_node=None, ends=(), defer=False, timeout=None), 'subtraction': StateNodeSpec(runnable=subtraction(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None, input_schema=<class 'dict'>, retry_policy=None, cache_policy=None, is_error_handler=False, error_handler_node=None, ends=(), defer=False, timeout=None)}
 +-----------+   
 | __start__ |   
 +-----------+   
        *        
        *        
        *        
  +----------+   
  | addition |   
  +----------+   
        *        
        *        
        *        
+-------------+  
| subtraction |  
+-------------+  
        *        
        *        
        *        
  +---------+    
  | __end__ |   

## 借助Pydantic构建稳定的State

### Pydantic基本使用
通过集成pydantic中的BaseModel抽象类来定义状态State, 定义后的状态可以对键值对属性进行自动校验，我们编写如下代码，对 a 和 b 键定义不同的类型，错误的类型会报错。

In [2]:
from pydantic import BaseModel
class MyState(BaseModel):
    a: int
    b: str="default"
# 自动校验
state = MyState(a=1)
print(state.a)
print(state.b)
# 类型错误会报错
state = MyState(a="aaa")
print(state.a)

1
default


ValidationError: 1 validation error for MyState
a
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='aaa', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing

### Pydantic应用于StateGraph

In [3]:
from langgraph.constants import START, END
from langgraph.graph import StateGraph
from pydantic import BaseModel

class CalcState(BaseModel):
    """
    定义计算过程中使用的状态模型

    属性:
        x (int): 用于传递和更新的整型数值
    """
    x: int
    
builder = StateGraph(CalcState)

def addition(state):
    """
    执行加法运算的节点函数

    参数:
        state (CalcState): 包含输入数据的状态对象，必须包含属性"x"

    返回:
        CalcState: 返回更新后的状态对象，其中"x"的值增加1
    """
    print(f'加法节点收到的初始值:{state}')
    return CalcState(x=state.x + 1)


def subtraction(state):
    """
    执行减法运算的节点函数

    参数:
        state (CalcState): 包含输入数据的状态对象，必须包含属性"x"

    返回:
        CalcState: 返回更新后的状态对象，其中"x"的值减少2
    """
    print(f'减法节点收到的初始值:{state}')
    return CalcState(x=state.x - 2)


# 向图构建器中添加节点
# 添加加法运算节点和减法运算节点到构建器中
builder.add_node("addition", addition)
builder.add_node("subtraction", subtraction)

# 定义节点之间的执行顺序 edges
# 设置节点间的依赖关系，形成执行流程图
builder.add_edge(START, "addition")
builder.add_edge("addition", "subtraction")
builder.add_edge("subtraction", END)

# 编译图构建器生成计算图
graph = builder.compile()

# 打印图的边和节点信息
print(builder.edges)
print(builder.nodes)

# 打印图的可视化结构
print(graph.get_graph().print_ascii())

# 定义一个初始状态对象，包含属性"x"为5
initial_state = CalcState(x=5)

# 调用graph对象的invoke方法，传入初始状态，执行图计算流程
result = graph.invoke(initial_state)

print(f"最后的结果是:{result}")


{('__start__', 'addition'), ('subtraction', '__end__'), ('addition', 'subtraction')}
{'addition': StateNodeSpec(runnable=addition(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None, input_schema=<class '__main__.CalcState'>, retry_policy=None, cache_policy=None, is_error_handler=False, error_handler_node=None, ends=(), defer=False, timeout=None), 'subtraction': StateNodeSpec(runnable=subtraction(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None, input_schema=<class '__main__.CalcState'>, retry_policy=None, cache_policy=None, is_error_handler=False, error_handler_node=None, ends=(), defer=False, timeout=None)}
 +-----------+   
 | __start__ |   
 +-----------+   
        *        
        *        
        *        
  +----------+   
  | addition |   
  +----------+   
        *        
        *        
        *        
+-------------+  
| subtraction |  
+-------------+  
        *        
        *        
        *        
  +---

## 流程控制语句
### 条件判断
使用langgraph构建了一个状态图，根据输入数值的奇偶性执行不同节点。check_x接收并传递状态，is_even判断奇偶，handle_even和handle_odd分别处理偶数和奇数情况，最终输出结果。

In [6]:
from typing import Optional
from langgraph.constants import START, END
from langgraph.graph import StateGraph
from loguru import logger
from pydantic import BaseModel


class MyState(BaseModel):
    """
    定义状态模型，用于在图节点之间传递数据
    
    Attributes:
        x (int): 输入的整数
        result (Optional[str]): 处理结果，可为"even"或"odd"
    """
    x: int
    result: Optional[str] = None


builder = StateGraph(MyState)


def check_x(state: MyState) -> MyState:
    """
    检查输入状态的节点函数
    
    Args:
        state (MyState): 包含输入数据的状态对象
        
    Returns:
        MyState: 返回原始状态对象，未做修改
    """
    logger.info(f"[check_x] Received state: {state}")
    return state


def is_even(state: MyState) -> bool:
    """
    判断状态中x值是否为偶数的条件函数
    
    Args:
        state (MyState): 包含待判断数值的状态对象
        
    Returns:
        bool: 如果x是偶数返回True，否则返回False
    """
    return state.x % 2 == 0


def handle_even(state: MyState) -> MyState:
    """
    处理偶数情况的节点函数
    
    Args:
        state (MyState): 包含偶数输入的状态对象
        
    Returns:
        MyState: 返回更新后的状态对象，result设置为"even"
    """
    logger.info("[handle_even] x 是偶数")
    return MyState(x=state.x, result="even")


def handle_odd(state: MyState) -> MyState:
    """
    处理奇数情况的节点函数
    
    Args:
        state (MyState): 包含奇数输入的状态对象
        
    Returns:
        MyState: 返回更新后的状态对象，result设置为"odd"
    """
    logger.info("[handle_odd] x 是奇数")
    return MyState(x=state.x, result="odd")


builder.add_node("check_x", check_x)
builder.add_node("handle_even", handle_even)
builder.add_node("handle_odd", handle_odd)


def is_even(state: MyState) -> bool:
    """
    判断状态中x值是否为偶数的条件函数
    
    Args:
        state (MyState): 包含待判断数值的状态对象
        
    Returns:
        bool: 如果x是偶数返回True，否则返回False
    """
    return state.x % 2 == 0


# 添加条件边，根据is_even函数的返回值决定流向哪个节点
builder.add_conditional_edges("check_x", is_even, {
    True: "handle_even",
    False: "handle_odd"
})

# 添加起始边，从START节点流向check_x节点
builder.add_edge(START, "check_x")

# 添加结束边，从处理节点流向END节点
builder.add_edge("handle_even", END)
builder.add_edge("handle_odd", END)

# 编译图结构
graph = builder.compile()

print("="*50)
print("状态机结构（ASCII纯文本）")
print(graph.get_graph().print_ascii())
print("="*50)

# 测试用例：输入偶数4
logger.info("输入 x=4（偶数）")
graph.invoke(MyState(x=4))

# 测试用例：输入奇数3
logger.info("输入 x=3（奇数）")
graph.invoke(MyState(x=3))

2026-05-18 14:42:24.011 | INFO     | __main__:<module>:118 - 输入 x=4（偶数）
2026-05-18 14:42:24.015 | INFO     | __main__:check_x:33 - [check_x] Received state: x=4 result=None
2026-05-18 14:42:24.018 | INFO     | __main__:handle_even:60 - [handle_even] x 是偶数
2026-05-18 14:42:24.030 | INFO     | __main__:<module>:122 - 输入 x=3（奇数）
2026-05-18 14:42:24.039 | INFO     | __main__:check_x:33 - [check_x] Received state: x=3 result=None
2026-05-18 14:42:24.042 | INFO     | __main__:handle_odd:74 - [handle_odd] x 是奇数


状态机结构（ASCII纯文本）
              +-----------+               
              | __start__ |               
              +-----------+               
                    *                     
                    *                     
                    *                     
               +---------+                
               | check_x |                
               +---------+                
             ...          ..              
            .               ..            
          ..                  ..          
+-------------+           +------------+  
| handle_even |           | handle_odd |  
+-------------+           +------------+  
             ***          **              
                *       **                
                 **   **                  
               +---------+                
               | __end__ |                
               +---------+                
None


{'x': 3, 'result': 'odd'}

### 循环语句
定义了一个基于循环流程，通过increment节点不断将状态中的x值加1，直到is_done条件判断x > 10成立时停止。初始x=6，每次执行increment节点更新状态，最终输出x=11。

In [12]:
from langgraph.constants import START, END
from langgraph.graph import StateGraph
from loguru import logger
from pydantic import BaseModel


class LoopState(BaseModel):
    """
    循环状态模型类
    
    Attributes:
        x (int): 状态变量，用于循环计数
    """
    x: int


builder = StateGraph(LoopState)


def increment(state: LoopState) -> LoopState:
    """
    增量函数，将状态中的x值加1
    
    Args:
        state (LoopState): 包含当前x值的循环状态对象
        
    Returns:
        LoopState: 返回更新后的循环状态对象，其中x值增加1
    """
    logger.info(f"[increment] 当前 x = {state.x}")
    return LoopState(x=state.x + 1)


builder.add_node("increment", increment)


def is_done(state: LoopState) -> bool:
    """
    判断循环是否结束的条件函数
    
    Args:
        state (LoopState): 包含当前x值的循环状态对象
        
    Returns:
        bool: 当x值大于10时返回True，否则返回False
    """
    return state.x > 10


builder.add_conditional_edges("increment", is_done, {
    True: END,
    False: "increment"
})

builder.add_edge(START, "increment")


# 编译图结构
graph = builder.compile()

# 执行循环
logger.info("执行循环直到 x > 10，初始x = 6")
final_state = graph.invoke(LoopState(x=6))

# 正确获取结果（Pydantic对象用 .x 访问）
logger.info(f"[最终结果] -> x = {final_state['x']}")
    

2026-05-18 14:49:02.798 | INFO     | __main__:<module>:62 - 执行循环直到 x > 10，初始x = 6
2026-05-18 14:49:02.807 | INFO     | __main__:increment:30 - [increment] 当前 x = 6
2026-05-18 14:49:02.809 | INFO     | __main__:increment:30 - [increment] 当前 x = 7
2026-05-18 14:49:02.812 | INFO     | __main__:increment:30 - [increment] 当前 x = 8
2026-05-18 14:49:02.817 | INFO     | __main__:increment:30 - [increment] 当前 x = 9
2026-05-18 14:49:02.826 | INFO     | __main__:increment:30 - [increment] 当前 x = 10
2026-05-18 14:49:02.828 | INFO     | __main__:<module>:66 - [最终结果] -> x = 11


### 判断循环复合图

In [14]:
from loguru import logger
from pydantic import BaseModel
from typing import Optional
from langgraph.graph import StateGraph, START, END


class BranchLoopState(BaseModel):
    """
    状态模型，用于保存当前流程中的变量状态。

    属性:
        x (int): 当前数值。
        done (Optional[bool]): 标记流程是否已完成，默认为 False。
    """
    x: int
    done: Optional[bool] = False


def check_x(state: BranchLoopState) -> BranchLoopState:
    """
    打印当前状态中 x 的值，用于调试和跟踪流程执行。

    参数:
        state (BranchLoopState): 包含当前 x 值的状态对象。

    返回:
        BranchLoopState: 返回未修改的原始状态对象。
    """
    logger.info(f"[check_x] 当前 x = {state.x}")
    return state


def is_even(state: BranchLoopState) -> bool:
    """
    判断当前状态中的 x 是否为偶数。

    参数:
        state (BranchLoopState): 包含当前 x 值的状态对象。

    返回:
        bool: 如果 x 是偶数则返回 True，否则返回 False。
    """
    return state.x % 2 == 0


def increment(state: BranchLoopState) -> BranchLoopState:
    """
    将当前状态中的 x 加一，并记录日志。

    参数:
        state (BranchLoopState): 包含当前 x 值的状态对象。

    返回:
        BranchLoopState: 返回更新后的状态对象（x+1）。
    """
    logger.info(f"[increment] x 是偶数，执行 +1 → {state.x + 1}")
    return BranchLoopState(x=state.x + 1)


def done(state: BranchLoopState) -> BranchLoopState:
    """
    标记流程完成，并记录日志。

    参数:
        state (BranchLoopState): 包含当前 x 值的状态对象。

    返回:
        BranchLoopState: 返回标记为完成的状态对象。
    """
    logger.info(f"[done] x 是奇数，流程结束")
    return BranchLoopState(x=state.x, done=True)


# 创建状态图并定义节点与边的关系
builder = StateGraph(BranchLoopState)
builder.add_node("check_x", check_x)
builder.add_node("increment", increment)
builder.add_node("done_node", done)

# 添加条件边：根据 is_even 函数的结果决定走向 increment 或 done_node
builder.add_conditional_edges("check_x", is_even, {
    True: "increment", False: "done_node"
})

# 定义流程路径：increment 节点之后回到 check_x 形成循环
builder.add_edge("increment", "check_x")

# 设置起始和结束节点连接
builder.add_edge(START, "check_x")
builder.add_edge("done_node", END)

# 编译状态图
graph = builder.compile()


# 测试用例1：从偶数开始，进入循环直到变为奇数
logger.info("初始 x=6（偶数，进入循环）")
final_state1 = graph.invoke(BranchLoopState(x=6))
logger.info("[最终结果1] ->", final_state1)

# 测试用例2：从奇数开始，直接结束流程
logger.info("初始 x=3（奇数，直接 done）")
final_state2 = graph.invoke(BranchLoopState(x=3))
logger.info("[最终结果2] ->", final_state2)


2026-05-18 14:50:57.257 | INFO     | __main__:<module>:97 - 初始 x=6（偶数，进入循环）
2026-05-18 14:50:57.269 | INFO     | __main__:check_x:29 - [check_x] 当前 x = 6
2026-05-18 14:50:57.272 | INFO     | __main__:increment:56 - [increment] x 是偶数，执行 +1 → 7
2026-05-18 14:50:57.274 | INFO     | __main__:check_x:29 - [check_x] 当前 x = 7
2026-05-18 14:50:57.278 | INFO     | __main__:done:70 - [done] x 是奇数，流程结束


2026-05-18 14:50:57.284 | INFO     | __main__:<module>:99 - [最终结果1] ->
2026-05-18 14:50:57.286 | INFO     | __main__:<module>:102 - 初始 x=3（奇数，直接 done）
2026-05-18 14:50:57.293 | INFO     | __main__:check_x:29 - [check_x] 当前 x = 3
2026-05-18 14:50:57.295 | INFO     | __main__:done:70 - [done] x 是奇数，流程结束
2026-05-18 14:50:57.298 | INFO     | __main__:<module>:104 - [最终结果2] ->


### 子图

In [17]:
from operator import add
from typing import TypedDict, Annotated
from langgraph.constants import END
from langgraph.graph import StateGraph, MessagesState, START

class State(TypedDict):
    """
    定义状态类，用于存储图节点间传递的消息状态
    messages: 使用add函数合并的字符串列表消息
    """
    messages: Annotated[list[str], add]

def sub_node(state:State) -> MessagesState:
    # 子图节点处理函数，接收当前状态并返回响应消息
    # @param state 当前状态对象，包含消息列表
    # @return 包含子图响应消息的新状态
    return {"messages": ["response from subgraph"]}

# 创建子图构建器并配置节点和边
subgraph_builder = StateGraph(State)
subgraph_builder.add_node("sub_node", sub_node)
subgraph_builder.add_edge(START, "sub_node")
subgraph_builder.add_edge("sub_node", END)
subgraph = subgraph_builder.compile()


# 创建主图构建器并添加子图节点
builder = StateGraph(State)
builder.add_node("subgraph_node", subgraph)
builder.add_edge(START, "subgraph_node")
builder.add_edge("subgraph_node", END)

# 编译主图并绘制结构图
graph = builder.compile()


# 执行图并打印结果
print(graph.invoke({"messages": ["hello subgraph"]}))


{'messages': ['hello subgraph', 'hello subgraph', 'response from subgraph']}
